# Guardião da Família — notebook do Colab

O Guardião avisa um familiar responsável quando um **número novo** diz ser parente de uma
pessoa idosa e pede dinheiro (o golpe do falso parente). Este notebook instala o backend,
lê as credenciais dos **Secrets do Colab** e roda o Guardião em um de dois modos:

- **demonstração** (padrão): a conta da pessoa idosa é simulada, com contatos, conversas
  antigas e números fictícios (DDD 00, que não existe no Brasil). O bot do Telegram que
  alerta a responsável continua real.
- **real**: lê a conta de teste do Telegram da pessoa idosa pelo Telethon.

Rode as células em ordem. Este é o esqueleto: filtro de contato novo, análise pelo Gemini,
casos e desfecho entram nas próximas versões.

**Segurança:** nenhuma credencial é escrita neste notebook. Elas ficam nos Secrets do Colab
(ícone de chave na barra lateral) e o notebook só mostra se cada uma está *presente* ou
*ausente*. Nunca cole um token ou chave numa célula.

## 1. Obter o código

A célula abaixo procura a pasta do projeto (com `backend/` e `frontend/`) nesta ordem:

1. **pasta existente** — a pasta atual, as pastas acima dela ou uma pasta em `/content`;
2. **ZIP enviado** — um `.zip` do projeto enviado para `/content` (painel *Arquivos*), que é extraído;
3. **`git clone`** do repositório público, só no Colab. Não precisa de token.

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

try:
    import google.colab  # noqa: F401

    EM_COLAB = True
except ImportError:
    EM_COLAB = False

REPOSITORIO = "https://github.com/fiap-poa/guardiao_gs_fiap.git"
CONTENT = Path("/content")
DESTINO_CLONE = CONTENT / "guardiao_gs_fiap"


def _eh_raiz(pasta):
    return (pasta / "backend" / "guardiao").is_dir() and (pasta / "frontend").is_dir()


def _procurar_raiz():
    candidatos = [Path.cwd(), *Path.cwd().parents]
    if CONTENT.is_dir():
        candidatos += sorted(p for p in CONTENT.glob("*") if p.is_dir())
        candidatos += sorted(p for p in CONTENT.glob("*/*") if p.is_dir())
    return next((p.resolve() for p in candidatos if _eh_raiz(p)), None)


def _extrair_zip():
    if not CONTENT.is_dir():
        return False
    for arquivo in sorted(CONTENT.glob("*.zip")):
        with zipfile.ZipFile(arquivo) as zip_:
            if any(nome.endswith("backend/guardiao/__init__.py") for nome in zip_.namelist()):
                destino = CONTENT / arquivo.stem
                zip_.extractall(destino)
                print(f"ZIP extraído: {arquivo.name} -> {destino}")
                return True
    return False


def _clonar():
    # repositório público: sem token; GIT_TERMINAL_PROMPT=0 faz o git falhar em vez de travar pedindo usuário
    ambiente = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    comando = ["git", "clone", "--depth", "1", "-q", REPOSITORIO, str(DESTINO_CLONE)]
    print("clonando o repositório...")
    try:
        resultado = subprocess.run(comando, capture_output=True, text=True, env=ambiente, timeout=120)
        codigo, erro = resultado.returncode, resultado.stderr.strip()
    except subprocess.TimeoutExpired:
        codigo, erro = None, "tempo esgotado (120 s)"
    if codigo != 0:
        raise RuntimeError(f"git clone falhou (código {codigo}): {erro}")


RAIZ = _procurar_raiz()
if RAIZ is None and _extrair_zip():
    RAIZ = _procurar_raiz()
if RAIZ is None:
    if EM_COLAB:
        _clonar()
        RAIZ = _procurar_raiz()
    else:
        print("aviso: fora do Colab o clone é pulado; abra este notebook de dentro do repositório")
if RAIZ is None:
    raise RuntimeError("não achei a pasta do projeto (com backend/ e frontend/)")

BACKEND = RAIZ / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
print(f"projeto em {RAIZ} (Colab: {'sim' if EM_COLAB else 'não'})")

## 2. Instalar as dependências

No Colab, instala `backend/requirements.txt` (Flask, Telethon, cliente HTTP da Bot API…).
Fora do Colab a instalação é pulada: use o ambiente virtual do backend
(`backend/.venv`) como kernel.

In [ ]:
import importlib.util

REQUISITOS = BACKEND / "requirements.txt"
if EM_COLAB:
    instalacao = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUISITOS)], capture_output=True, text=True
    )
    if instalacao.returncode != 0:
        print("\n".join(instalacao.stderr.strip().splitlines()[-20:]))
        raise RuntimeError(f"pip install falhou (código {instalacao.returncode})")
    print("dependências instaladas")
else:
    faltando = [m for m in ("flask", "telethon", "requests", "dotenv", "qrcode") if importlib.util.find_spec(m) is None]
    if faltando:
        print(f"aviso: fora do Colab a instalação é pulada; faltam {faltando}: pip install -r {REQUISITOS}")
    else:
        print("aviso: fora do Colab a instalação é pulada (as dependências já estão no kernel)")

## 3. Credenciais (Secrets do Colab)

No Colab, abra o ícone de **chave** na barra lateral, crie os Secrets abaixo e ative
*Acesso ao notebook* em cada um:

| Secret | Para quê | Modo |
|---|---|---|
| `TELEGRAM_BOT_TOKEN` | bot do Guardião (alerta a responsável) | demonstração e real |
| `GEMINI_API_KEY` | análise das mensagens (próximas versões) | demonstração e real |
| `TELEGRAM_API_ID`, `TELEGRAM_API_HASH` | ler a conta de teste pelo Telethon | só real |

A célula copia os Secrets para variáveis de ambiente do processo e mostra só
**presente/ausente**. Fora do Colab, as credenciais vêm de `backend/.env` (fora do git).
Sem `TELEGRAM_BOT_TOKEN`, o modo demonstração avisa e segue sem bot: os cenários rodam,
mas nenhum alerta chega à responsável.

In [ ]:
from guardiao.config import carregar_env, carregar_segredos_colab

SEGREDOS = ["TELEGRAM_BOT_TOKEN", "GEMINI_API_KEY", "TELEGRAM_API_ID", "TELEGRAM_API_HASH"]
do_colab = carregar_segredos_colab(SEGREDOS)
if not EM_COLAB:
    print("aviso: fora do Colab não há Secrets; lendo backend/.env\n")
    carregar_env()
for nome in SEGREDOS:
    presente = do_colab[nome] or bool(os.environ.get(nome, "").strip())
    print(f"{nome:20} {'presente' if presente else 'ausente'}")
if not os.environ.get("TELEGRAM_BOT_TOKEN", "").strip():
    print("\naviso: sem TELEGRAM_BOT_TOKEN o modo demonstração segue sem bot (nenhum alerta sai)")

## 4. Escolher o modo

- `"demonstracao"`: canal simulado. É o modo da entrega e não precisa de conta do Telegram.
- `"real"`: Telethon numa conta **de teste** do Telegram (nunca a conta de uma pessoa real).
  Precisa de `TELEGRAM_API_ID`, `TELEGRAM_API_HASH` e de uma sessão já logada por QR;
  é opcional.

In [ ]:
MODO = "demonstracao"  # ou "real"

from guardiao.canal.fabrica import MODOS

if MODO not in MODOS:
    raise ValueError(f"MODO deve ser um de {MODOS}")
print(f"modo: {MODO}")

## 5. Criar o canal

O canal é a única parte que fala com o mensageiro. O `handler` abaixo guarda tudo o que o
canal entrega em `RECEBIDOS`: mensagens que chegam à conta da pessoa idosa e, pelo bot,
os toques em Iniciar e nos botões dos alertas. Rodar a célula de novo fecha o canal
anterior antes de criar outro.

In [ ]:
from guardiao.canal.fabrica import criar_canal
from guardiao.canal.simulado import descrever_recebido

if globals().get("canal") is not None:
    canal.fechar()
canal = None  # se a criação falhar, não sobra referência ao canal fechado

RECEBIDOS = []


def handler(evento):
    RECEBIDOS.append(evento)


canal = criar_canal(MODO)
try:
    canal.receber_mensagens(handler)
except Exception as erro:
    print(f"aviso: o canal não começou a receber ({type(erro).__name__})")
print(canal)

## 6. Cenários do modo demonstração

A pessoa idosa simulada é a **Dona Maria**. Os contatos e conversas antigas são fictícios,
com DDD 00. Os cinco cenários:

| Cenário | O que acontece | Esperado |
|---|---|---|
| `golpe_classico` | número novo diz ser o neto Pedro e pede Pix | alerta |
| `parente_trocou_numero` | o filho Lucas trocou de número de verdade e pede transferência | alerta, e a responsável responde "Era ele mesmo" |
| `contato_antigo_pede_dinheiro` | o vizinho Carlos, que já conversava com ela, pede dinheiro | ignorado |
| `mensagem_em_grupo` | alguém no grupo Família pede Pix | ignorado (o Guardião não lê grupos) |
| `contato_novo_sem_dinheiro` | número novo diz ser a neta Júlia e só manda saudades | sem alerta |

In [ ]:
from guardiao.canal.cenarios import CENARIOS

for cenario in CENARIOS:
    print(f"{cenario.id:30} esperado={cenario.esperado:22} {cenario.descricao}")

if MODO == "demonstracao":
    print("\nContatos salvos no celular simulado:")
    for contato in canal.listar_contatos():
        print(f"  {contato.nome:16} {contato.telefone}")
    print("\nConversas que já existiam:")
    for conversa in canal.conversas_existentes():
        print(f"  {conversa.nome:16} ({conversa.tipo})")

### Injetar os cinco cenários

Cada mensagem do cenário chega ao canal simulado como se tivesse chegado ao Telegram da
Dona Maria. A célula mostra o que o handler recebeu: em grupo, só o tipo, sem remetente
nem texto, como no canal Telegram.

In [ ]:
if MODO != "demonstracao":
    print("aviso: a injeção de cenários só existe no modo demonstração; pulando")
else:
    for cenario in CENARIOS:
        print(f"== {cenario.id} (esperado: {cenario.esperado})")
        for evento in canal.injetar(cenario.id):
            print("   handler recebeu:", descrever_recebido(evento))

### Celular simulado da Dona Maria

Mostra o que chegou ao mensageiro dela e os avisos do Guardião (marcados com `>>`).
O aviso à pessoa idosa nunca sai pelo bot no modo demonstração: aparece aqui. A célula
manda um aviso de teste só para mostrar o celular; o aviso automático (golpe confirmado ou
10 minutos sem resposta) vem numa próxima versão.

In [ ]:
from guardiao.canal.simulado import formatar_celular

if MODO != "demonstracao":
    print("aviso: o celular simulado só existe no modo demonstração; pulando")
else:
    canal.enviar_aviso(canal.id_idosa, "Teste do Guardião: este é o celular simulado da Dona Maria.")
    print(formatar_celular(canal.celular()))

### Eventos do bot

Com `TELEGRAM_BOT_TOKEN`, o canal escuta o bot. Abra o bot no seu Telegram, toque em
**Iniciar** e rode esta célula de novo: o `InicioBot` aparece abaixo (sem ids de usuário).

In [ ]:
from guardiao.canal.base import MensagemRecebida

bot = canal.saida if MODO == "demonstracao" else canal
if bot is None:
    print("aviso: modo demonstração sem bot (TELEGRAM_BOT_TOKEN ausente); pulando")
else:
    try:
        print(f"bot: @{bot.info_bot().get('username')}")
    except Exception as erro:
        print(f"aviso: não consegui falar com o bot ({type(erro).__name__})")
    eventos = [e for e in RECEBIDOS if not isinstance(e, MensagemRecebida)]
    for evento in eventos:
        print("handler recebeu:", descrever_recebido(evento))
    if not eventos:
        print("(nenhum Iniciar ou toque recebido ainda)")

## 7. Tela do familiar

Sobe o servidor Flask numa thread e mostra a tela dentro do notebook com
`serve_kernel_port_as_iframe`. A tela lê `api/estado` por caminho relativo, o que funciona
no iframe do Colab. Fora do Colab, abra o endereço impresso no navegador.

In [ ]:
from guardiao.app import iniciar_em_segundo_plano

try:
    servidor = iniciar_em_segundo_plano()
except OSError as erro:
    servidor = None
    print(f"aviso: não consegui subir o Flask ({type(erro).__name__}: porta em uso?)")
if servidor is not None:
    print(f"Flask no ar na porta {servidor.port}")
    if EM_COLAB:
        from google.colab import output

        output.serve_kernel_port_as_iframe(servidor.port, path="/", height=420)
    else:
        print(f"aviso: fora do Colab não há iframe; abra http://127.0.0.1:{servidor.port}/")

## 8. Encerrar

Para o polling do bot, desconecta o canal e desliga a tela. Rode ao terminar a demonstração.

In [ ]:
from guardiao.app import parar_servidor

if globals().get("canal") is not None:
    canal.fechar()
    canal = None
parar_servidor()
print("Guardião encerrado")